In [ ]:
# modele optimisé pour la mémoire
import os
import numpy as np
import pandas as pd
import gc
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score

import matplotlib.pyplot as plt
import seaborn as sns

print(" Imports terminés")

In [ ]:
# chargement des données

data_dir = 'data/'

# Chargement des fichiers
app_train = pd.read_csv(data_dir + 'application_train.csv')
app_test = pd.read_csv(data_dir + 'application_test.csv')
bureau = pd.read_csv(data_dir + 'bureau.csv')
bureau_balance = pd.read_csv(data_dir + 'bureau_balance.csv')
credit_card = pd.read_csv(data_dir + 'credit_card_balance.csv')
installments = pd.read_csv(data_dir + 'installments_payments.csv')
previous = pd.read_csv(data_dir + 'previous_application.csv')
pos_cash = pd.read_csv(data_dir + 'POS_CASH_balance.csv')

print(f"Train: {app_train.shape}, Test: {app_test.shape}")
print(" Fichiers chargés")

In [ ]:
# features enginnering fichiers train et test de kaggle
# Anomalies DAYS_EMPLOYED
app_train['DAYS_EMPLOYED_ANOM'] = app_train["DAYS_EMPLOYED"] == 365243
app_test['DAYS_EMPLOYED_ANOM'] = app_test["DAYS_EMPLOYED"] == 365243
app_train['DAYS_EMPLOYED'].replace({365243: np.nan}, inplace=True)
app_test['DAYS_EMPLOYED'].replace({365243: np.nan}, inplace=True)

# Ratios
app_train['CREDIT_INCOME_RATIO'] = app_train['AMT_CREDIT'] / app_train['AMT_INCOME_TOTAL']
app_test['CREDIT_INCOME_RATIO'] = app_test['AMT_CREDIT'] / app_test['AMT_INCOME_TOTAL']
app_train['ANNUITY_INCOME_RATIO'] = app_train['AMT_ANNUITY'] / app_train['AMT_INCOME_TOTAL']
app_test['ANNUITY_INCOME_RATIO'] = app_test['AMT_ANNUITY'] / app_test['AMT_INCOME_TOTAL']
app_train['CREDIT_TERM'] = app_train['AMT_ANNUITY'] / app_train['AMT_CREDIT']
app_test['CREDIT_TERM'] = app_test['AMT_ANNUITY'] / app_test['AMT_CREDIT']
app_train['DAYS_BIRTH'] = abs(app_train['DAYS_BIRTH'])
app_test['DAYS_BIRTH'] = abs(app_test['DAYS_BIRTH'])

print(" Features créées")

In [ ]:
# Agrégation

# BUREAU + BUREAU_BALANCE  
bureau_balance_agg = bureau_balance.groupby('SK_ID_BUREAU').agg({
    'MONTHS_BALANCE': ['min', 'max', 'size']
})
bureau_balance_agg.columns = ['_'.join(col) for col in bureau_balance_agg.columns]
bureau_balance_agg.reset_index(inplace=True)

bureau_fusion = bureau.merge(bureau_balance_agg, on='SK_ID_BUREAU', how='left')
bureau_agg = bureau_fusion.groupby('SK_ID_CURR').agg(['mean', 'max', 'sum'])
bureau_agg.columns = ['bureau_' + '_'.join(col).upper() for col in bureau_agg.columns]
bureau_agg.reset_index(inplace=True)

print(f" Bureau: {bureau_agg.shape}")
del bureau, bureau_balance, bureau_balance_agg, bureau_fusion
gc.collect()

In [ ]:
# CREDIT CARD
credit_card_agg = credit_card.groupby('SK_ID_PREV').agg({
    'AMT_BALANCE': ['mean', 'max'],
    'AMT_CREDIT_LIMIT_ACTUAL': ['mean'],
    'SK_DPD': ['mean', 'max']
})
credit_card_agg.columns = ['CC_' + '_'.join(col).upper() for col in credit_card_agg.columns]
credit_card_agg.reset_index(inplace=True)
credit_card_client = credit_card[['SK_ID_PREV', 'SK_ID_CURR']].drop_duplicates()
credit_card_client = credit_card_client.merge(credit_card_agg, on='SK_ID_PREV', how='left')

print(f" Credit card: {credit_card_client.shape}")
del credit_card, credit_card_agg
gc.collect()

In [ ]:
# INSTALLMENTS (version simplifiée)
installments['PAYMENT_DIFF'] = installments['AMT_PAYMENT'] - installments['AMT_INSTALMENT']
installments['LATE_PAYMENT'] = (installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT'] > 0).astype(int)

installments_agg = installments.groupby('SK_ID_PREV').agg({
    'AMT_PAYMENT': ['mean', 'sum'],
    'PAYMENT_DIFF': ['mean'],
    'LATE_PAYMENT': ['sum', 'mean']
})
installments_agg.columns = ['INSTAL_' + '_'.join(col).upper() for col in installments_agg.columns]
installments_agg.reset_index(inplace=True)

# Fusion Credit + Installments
credit_prev_merged = credit_card_client.merge(installments_agg, on='SK_ID_PREV', how='outer')
credit_client_final = credit_prev_merged.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max'])
credit_client_final.columns = ['CREDIT_' + '_'.join(col).upper() for col in credit_client_final.columns]
credit_client_final.reset_index(inplace=True)

print(f" Credit+Installments: {credit_client_final.shape}")
del installments, installments_agg, credit_card_client, credit_prev_merged
gc.collect()

In [ ]:
# PREVIOUS 
prev_agg = previous.groupby('SK_ID_CURR').agg({
    'AMT_ANNUITY': ['mean', 'max'],
    'AMT_CREDIT': ['mean', 'max'],
    'AMT_APPLICATION': ['mean'],
    'DAYS_DECISION': ['mean'],
    'CNT_PAYMENT': ['mean', 'sum']
})
prev_agg.columns = ['PREV_' + '_'.join(col).upper() for col in prev_agg.columns]
prev_agg.reset_index(inplace=True)

print(f" Previous: {prev_agg.shape}")
del previous
gc.collect()

In [ ]:
# POS_CASH (version simplifiée)
pos_agg = pos_cash.groupby('SK_ID_CURR').agg({
    'MONTHS_BALANCE': ['min', 'max'],
    'CNT_INSTALMENT': ['mean', 'max'],
    'SK_DPD': ['mean', 'max', 'sum']
})
pos_agg.columns = ['POS_' + '_'.join(col).upper() for col in pos_agg.columns]
pos_agg.reset_index(inplace=True)

print(f" POS_CASH: {pos_agg.shape}")
del pos_cash
gc.collect()

In [ ]:
# Fusion avec train et test
train = app_train.copy()
test = app_test.copy()

train = train.merge(bureau_agg, on='SK_ID_CURR', how='left')
test = test.merge(bureau_agg, on='SK_ID_CURR', how='left')
train = train.merge(credit_client_final, on='SK_ID_CURR', how='left')
test = test.merge(credit_client_final, on='SK_ID_CURR', how='left')
train = train.merge(prev_agg, on='SK_ID_CURR', how='left')
test = test.merge(prev_agg, on='SK_ID_CURR', how='left')
train = train.merge(pos_agg, on='SK_ID_CURR', how='left')
test = test.merge(pos_agg, on='SK_ID_CURR', how='left')

print(f" Fusion terminée - Train: {train.shape}, Test: {test.shape}")
del bureau_agg, credit_client_final, prev_agg, pos_agg
gc.collect()

In [ ]:
# Encodage
# Label Encoding
categorical_columns = train.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()

for col in categorical_columns:
    if train[col].nunique() <= 2:
        train[col] = train[col].fillna('Missing').astype(str)
        test[col] = test[col].fillna('Missing').astype(str)
        combined = pd.concat([train[col], test[col]])
        le.fit(combined)
        train[col] = le.transform(train[col])
        test[col] = le.transform(test[col])

# One-Hot limité (max 10 catégories par variable)
for col in categorical_columns:
    if col in train.columns and train[col].dtype == 'object':
        if train[col].nunique() <= 10:
            train = pd.concat([train, pd.get_dummies(train[col], prefix=col)], axis=1)
            test = pd.concat([test, pd.get_dummies(test[col], prefix=col)], axis=1)
        train = train.drop(columns=[col])
        test = test.drop(columns=[col])

# Aligner
train_labels = train['TARGET']
train = train.drop(columns=['TARGET'])
train, test = train.align(test, join='inner', axis=1)
train['TARGET'] = train_labels

print(f" Encodage - Train: {train.shape}, Test: {test.shape}")

In [ ]:
# Optimisation
# Supprimer colonnes avec >90% NaN
threshold = 0.90
missing = (train.isnull().sum() / len(train))
cols_drop = missing[missing > threshold].index.tolist()
train = train.drop(columns=cols_drop)
test = test.drop(columns=cols_drop)

# Supprimer colonnes à variance nulle
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
for col in ['TARGET', 'SK_ID_CURR']:
    if col in numeric_cols:
        numeric_cols.remove(col)

low_var = [col for col in numeric_cols if train[col].nunique() == 1]
train = train.drop(columns=low_var)
test = test.drop(columns=low_var)

# Sélection top 200 features par corrélation
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
for col in ['TARGET', 'SK_ID_CURR']:
    if col in numeric_cols:
        numeric_cols.remove(col)

corr = train[numeric_cols].corrwith(train['TARGET']).abs().sort_values(ascending=False)
top_features = corr.head(200).index.tolist()
train = train[['SK_ID_CURR', 'TARGET'] + top_features]
test = test[['SK_ID_CURR'] + top_features]

print(f" Optimisé - Train: {train.shape}, Test: {test.shape}")

In [ ]:
# préparation du modele
X = train.drop(columns=['TARGET', 'SK_ID_CURR'])
y = train['TARGET']
X_test = test.drop(columns=['SK_ID_CURR'])

# Remplacer NaN par -999 (plus rapide que l'imputation)
X = X.fillna(-999)
X_test = X_test.fillna(-999)

print(f" X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")

In [ ]:
# modélisation simple

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
cv_lr = cross_val_score(lr, X, y, cv=5, scoring='roc_auc')
print(f"Logistic Regression - ROC AUC: {cv_lr.mean():.4f} (+/- {cv_lr.std():.4f})")

lr.fit(X, y)
pred_lr = lr.predict_proba(X_test)[:, 1]

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
cv_rf = cross_val_score(rf, X, y, cv=3, scoring='roc_auc')
print(f"Random Forest - ROC AUC: {cv_rf.mean():.4f} (+/- {cv_rf.std():.4f})")

rf.fit(X, y)
pred_rf = rf.predict_proba(X_test)[:, 1]

print(" Modèles entraînés")

In [ ]:
# LightGBM

In [ ]:
# CatBoost

In [ ]:
# XGBoost

In [ ]:
# Sauvegarde

pd.DataFrame({'SK_ID_CURR': test['SK_ID_CURR'], 'TARGET': pred_lr}).to_csv('submission_lr.csv', index=False)
pd.DataFrame({'SK_ID_CURR': test['SK_ID_CURR'], 'TARGET': pred_rf}).to_csv('submission_rf.csv', index=False)

print("✅ Fichiers sauvegardés")
print(f"\n📊 ROC AUC - LR: {cv_lr.mean():.4f}, RF: {cv_rf.mean():.4f}")